# Notebook 03: Model Inference & Ensemble Output

**Purpose**: Load saved models, run inference on inventory data, compute combined
Replacement Score, apply color-coded priority ranking, and export results.

**Outputs**:
- `replacement_priority.csv` — ranked list of all assets with predictions
- Color-coded priority dashboard display

In [ ]:
# === IMPORTS ===
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# === CONFIGURATION ===
CURRENT_YEAR = 2026
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data'
MODEL_DIR = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'

print(f'Base:   {BASE_DIR}')
print(f'Data:   {DATA_DIR}')
print(f'Models: {MODEL_DIR}')
print(f'Output: {OUTPUT_DIR}')

---
## Step 1: Load Saved Models

In [ ]:
def load_models():
    try:
        clf = joblib.load(MODEL_DIR / 'replacement_model.pkl')
        reg = joblib.load(MODEL_DIR / 'health_score_model.pkl')
        encoder = joblib.load(MODEL_DIR / 'label_encoder.pkl')
        feature_cols = joblib.load(MODEL_DIR / 'feature_columns.pkl')
        print('All models loaded successfully')
        print(f'  Classifier: {type(clf).__name__}')
        print(f'  Regressor:  {type(reg).__name__}')
        print(f'  Classes:    {list(encoder.classes_)}')
        print(f'  Features:   {len(feature_cols)}')
        return clf, reg, encoder, feature_cols
    except Exception as e:
        print(f'Error loading models: {e}')
        print('Run Notebook 02 to train models first.')
        raise

clf, reg, label_encoder, feature_columns = load_models()

---
## Step 2: Load & Preprocess Inventory Data

We repeat the same feature engineering from Notebook 01 so we can run inference
on any new ICT asset data.

In [ ]:
def load_inventory():
    try:
        inv = pd.read_csv(DATA_DIR / 'inv_inventory.csv', low_memory=False)
        repair = pd.read_csv(DATA_DIR / 'repairhistory.csv', low_memory=False)
        print(f'Loaded {len(inv)} inventory records, {len(repair)} repair records')
        return inv, repair
    except Exception as e:
        print(f'Error loading data: {e}')
        raise

inv, repair = load_inventory()

In [ ]:
# --- Merge & Aggregate ---
inv['id'] = pd.to_numeric(inv['id'], errors='coerce')
repair['Equiment ID'] = pd.to_numeric(repair['Equiment ID'], errors='coerce')

repair_agg = repair.groupby('Equiment ID').agg(
    total_repairs=('SRF Track ID', 'count'),
    latest_repair_date=('Date Recorded', 'max'),
    last_repair_remarks=('Remarks / Action Taken', lambda x: x.dropna().iloc[-1] if not x.dropna().empty else 'No remarks recorded'),
    repair_frequency=('Date Recorded', 'nunique')
).reset_index()
repair_agg.columns = ['id', 'total_repairs', 'latest_repair_date', 'last_repair_remarks', 'repair_frequency']

inv = inv.merge(repair_agg, on='id', how='left')
inv['total_repairs'] = inv['total_repairs'].fillna(0).astype(int)
inv['repair_frequency'] = inv['repair_frequency'].fillna(0).astype(int)
inv['latest_repair_date'] = inv['latest_repair_date'].fillna('No repairs')
inv['last_repair_remarks'] = inv['last_repair_remarks'].fillna('No repairs')
print('Merge and aggregation complete')

### 2.1 Feature Engineering Functions

Reusable functions matching Notebook 01.

In [ ]:
def extract_specs(specs):
    if pd.isna(specs) or specs is None:
        return pd.Series({'cpu': 'Unknown', 'ram_gb': 0, 'storage_gb': 0, 'has_ssd': 0, 'has_gpu': 0})
    specs = str(specs)
    cpu = 'Unknown'
    for p in [r'(Intel (Core )?[iI]\d[- ][\w\d]+)', r'(Ryzen [\d\s\w]+)', r'(Core [iI]\d)']:
        m = re.search(p, specs)
        if m:
            cpu = m.group(1).strip()
            break
    ram = 0
    ram_match = re.search(r'(\d+)\s*GB\s*(?:RAM|ram|DDR[\d]?|memory)', specs)
    if ram_match:
        ram = int(ram_match.group(1))
    if ram <= 8:
        if '8GB+8GB' in specs.replace(' ', ''):
            ram = 16
    storage = 0
    for s in re.findall(r'(\d+)\s*(?:GB|gb|TB|tb)', specs):
        val = int(s)
        if 'TB' in specs or 'tb' in specs:
            val *= 1024
        if val >= 120:
            storage = val
            break
    has_ssd = 1 if re.search(r'SSD|ssd|NVMe|nvme', specs) else 0
    has_gpu = 1 if re.search(r'GeForce|GTX|RTX|Radeon|NVIDIA|MX\d{3}', specs) else 0
    return pd.Series({'cpu': cpu, 'ram_gb': ram, 'storage_gb': storage, 'has_ssd': has_ssd, 'has_gpu': has_gpu})

def compute_age_features(df):
    df['yearAcquired'] = pd.to_numeric(df['yearAcquired'], errors='coerce')
    df['equipment_age'] = (CURRENT_YEAR - df['yearAcquired']).clip(lower=0)
    shelf_map = {'Within 5 Year': 5, 'Beyond 5 year': 5, 'within5years': 5}
    df['shelf_life_years'] = df['shelfLife'].map(shelf_map).fillna(5)
    df['remaining_useful_life'] = (df['shelf_life_years'] - df['equipment_age']).clip(lower=0)
    return df

def compute_depreciation_and_license(df):
    df['depreciation_pct'] = (df['equipment_age'] / df['shelf_life_years'] * 100).clip(upper=100)
    license_map = {'Evaluation Copy': 2, 'Evaluation Copy ': 2, 'EVALUATION COPY': 2,
                   'Perpetual Copy': 0, 'Perpetual Copy ': 0}
    df['license_risk'] = df['licensingModel'].map(license_map).fillna(1)
    return df

def compute_performance_score(df):
    def _score(row):
        s = 0
        if row['ram_gb'] >= 16: s += 30
        elif row['ram_gb'] >= 8: s += 20
        elif row['ram_gb'] >= 4: s += 10
        cpu_str = str(row['cpu']).lower()
        if any(x in cpu_str for x in ['i7', 'i9', 'ryzen 7', 'ryzen 9']): s += 30
        elif any(x in cpu_str for x in ['i5', 'ryzen 5']): s += 20
        elif any(x in cpu_str for x in ['i3', 'ryzen 3']): s += 10
        else: s += 5
        s += 25 if row['has_ssd'] else 0
        s += 15 if row['has_gpu'] else 0
        return min(s, 100)
    return df.apply(_score, axis=1)

def compute_health_score(df):
    max_age = max(df['equipment_age'].max(), 1)
    max_rep = max(df['total_repairs'].max(), 1)
    age_sc = ((1 - df['equipment_age'] / max_age) * 100).clip(0, 100)
    rep_sc = ((1 - df['total_repairs'] / max_rep) * 100).clip(0, 100)
    perf_sc = df['performance_score']
    dep_sc = (100 - df['depreciation_pct']).clip(0, 100)
    lic_sc = ((1 - df['license_risk'] / 2) * 100).clip(0, 100)
    remarks_sc = df['remarks'].apply(
        lambda r: 20 if pd.notna(r) and any(w in str(r).lower() for w in ['unserviceable', 'replace', 'beyond', 'failing', 'broken'])
        else 80 if pd.notna(r) and any(w in str(r).lower() for w in ['operational', 'working', 'functional'])
        else 50)
    return (age_sc * 0.30 + rep_sc * 0.25 + perf_sc * 0.20 + dep_sc * 0.10 + lic_sc * 0.10 + remarks_sc * 0.05).round(2).fillna(50)

In [ ]:
# Apply feature engineering
specs_df = inv['specifications'].apply(extract_specs)
inv = pd.concat([inv, specs_df], axis=1)
inv = compute_age_features(inv)
inv = compute_depreciation_and_license(inv)
inv['performance_score'] = compute_performance_score(inv)
inv['asset_health_score'] = compute_health_score(inv)
print('Feature engineering complete')
print(f'Inventory shape: {inv.shape}')

---
## Step 3: Prepare Feature Matrix for Prediction

In [ ]:
def prepare_inference_features(df, feature_cols):
    features = pd.DataFrame()
    features['equipment_age'] = df['equipment_age']
    features['total_repairs'] = df['total_repairs']
    features['repair_frequency'] = df['repair_frequency']
    features['remaining_useful_life'] = df['remaining_useful_life']
    features['depreciation_pct'] = df['depreciation_pct']
    features['license_risk'] = df['license_risk']
    features['performance_score'] = df['performance_score']
    features['ram_gb'] = df['ram_gb']
    features['storage_gb'] = df['storage_gb']
    features['has_ssd'] = df['has_ssd']
    features['has_gpu'] = df['has_gpu']

    # One-hot encoding with alignment to training columns
    type_dummies = pd.get_dummies(df['equipmentType'], prefix='type')
    brand_clean = df['brand'].fillna('Unknown').str.strip().replace(['', 'N/A', '0'], 'Unknown')
    brand_dummies = pd.get_dummies(brand_clean, prefix='brand')
    status_clean = df['statusOfEmployment'].fillna('Unknown').str.strip()
    status_dummies = pd.get_dummies(status_clean, prefix='status')
    nature_clean = df['natureOfWork'].fillna('Unknown').str.strip()
    nature_dummies = pd.get_dummies(nature_clean, prefix='nature')
    div_clean = df['officeDivision'].fillna('Unknown').str.strip()
    div_dummies = pd.get_dummies(div_clean, prefix='division')

    features = pd.concat([
        features, type_dummies, brand_dummies,
        status_dummies, nature_dummies, div_dummies
    ], axis=1)

    features = features.fillna(0)

    # Align columns to match training feature_columns
    for col in feature_cols:
        if col not in features.columns:
            features[col] = 0
    features = features[feature_cols]

    return features

X_inference = prepare_inference_features(inv, feature_columns)
print(f'Inference feature matrix: {X_inference.shape}')
print(f'Columns match training: {list(X_inference.columns) == feature_columns}')

---
## Step 4: Run Dual Model Inference

In [ ]:
# Model A: Classifier — predict Replacement Priority
y_pred_class = clf.predict(X_inference)
y_pred_proba = clf.predict_proba(X_inference)
priority_labels = label_encoder.inverse_transform(y_pred_class)
max_proba = y_pred_proba.max(axis=1)

# Model B: Regressor — predict Health Score
y_pred_health = reg.predict(X_inference)
y_pred_health = np.clip(y_pred_health, 0, 100).round(2)

print('Inference complete')
print(f'\nPriority distribution:')
print(pd.Series(priority_labels).value_counts())
print(f'\nHealth Score stats:')
print(f'  Mean: {y_pred_health.mean():.2f}')
print(f'  Min:  {y_pred_health.min():.2f}')
print(f'  Max:  {y_pred_health.max():.2f}')

---
## Step 5: Compute Replacement Score (Ensemble Output)

Replace Score = blend of classifier probability confidence × regressor health score
normalized to 0-100.

In [ ]:
def compute_replacement_score(priority_labels, max_proba, health_scores):
    # Base score from priority
    priority_base = np.array([
        90 if p == 'Critical' else
        70 if p == 'High' else
        50 if p == 'Medium' else
        20 for p in priority_labels
    ])

    # Inverse health (lower health = higher replacement need)
    inverse_health = 100 - health_scores

    # Blend: 60% priority_base + 40% inverse_health, modulated by confidence
    replacement_score = (priority_base * 0.6 + inverse_health * 0.4) * max_proba
    replacement_score = replacement_score.round(2)
    return replacement_score

inv['predicted_priority'] = priority_labels
inv['prediction_probability'] = max_proba.round(4)
inv['predicted_health_score'] = y_pred_health
inv['replacement_score'] = compute_replacement_score(
    priority_labels, max_proba, y_pred_health
)

---
## Step 6: Color-Coded Priority

In [ ]:
def get_color_and_label(score):
    if score >= 90:
        return '🔴 Replace Immediately', 'Replace Immediately'
    elif score >= 70:
        return '🟠 High', 'High'
    elif score >= 50:
        return '🟡 Medium', 'Medium'
    else:
        return '🟢 Low', 'Low'

color_labels = inv['replacement_score'].apply(get_color_and_label)
inv['priority_color'] = color_labels.apply(lambda x: x[0])
inv['priority_rank_label'] = color_labels.apply(lambda x: x[1])

print('Color coding applied')
print(inv['priority_rank_label'].value_counts())

---
## Step 7: Generate Recommendation

In [ ]:
def generate_recommendation(row):
    score = row['replacement_score']
    priority = row['predicted_priority']
    health = row['predicted_health_score']
    if score >= 90 or priority == 'Critical':
        return 'Immediate replacement required'
    elif score >= 70 or priority == 'High':
        return 'Plan replacement within 6 months'
    elif score >= 50 or priority == 'Medium':
        return 'Monitor and plan for next budget cycle'
    else:
        return 'No immediate action needed'

inv['recommendation'] = inv.apply(generate_recommendation, axis=1)

---
## Step 8: Export Replacement Priority CSV

In [ ]:
# Sort by replacement score descending
inv_sorted = inv.sort_values('replacement_score', ascending=False).reset_index(drop=True)
inv_sorted['rank'] = range(1, len(inv_sorted) + 1)

export_cols = [
    'rank', 'propertyNumber', 'equipmentType', 'brand', 'equipment_age',
    'total_repairs', 'asset_health_score', 'predicted_priority',
    'prediction_probability', 'replacement_score', 'priority_rank_label', 'recommendation',
    'office', 'accountablePerson', 'actualUser', 'specifications'
]

# Filter to available columns
export_cols = [c for c in export_cols if c in inv_sorted.columns]
replacement_priority = inv_sorted[export_cols]

replacement_priority.to_csv(OUTPUT_DIR / 'replacement_priority.csv', index=False)
print(f'Exported {len(replacement_priority)} records to replacement_priority.csv')
replacement_priority.head(20)

---
## Step 9: Summary Dashboard View

In [ ]:
print('=' * 80)
print('AI-DSS INFERENCE RESULTS SUMMARY')
print('=' * 80)
print(f'\nTotal Assets Analyzed:      {len(inv_sorted)}')
print(f'\n--- Replacement Priority ---')
print(inv_sorted['predicted_priority'].value_counts().to_string())
print(f'\n--- Health Categories ---')
health_cat = inv_sorted['predicted_health_score'].apply(
    lambda x: 'Excellent' if x >= 90 else 'Good' if x >= 80 else 'Aging' if x >= 60 else 'Poor' if x >= 40 else 'Replace'
)
print(health_cat.value_counts().to_string())
print(f'\n--- Top 5 Critical Assets ---')
critical = inv_sorted[inv_sorted['predicted_priority'] == 'Critical'].head(5)
if len(critical) > 0:
    print(critical[['propertyNumber', 'equipmentType', 'equipment_age', 'asset_health_score',
                    'replacement_score']].to_string(index=False))
else:
    print('No critical assets found.')

---
## Summary

Notebook 03 completed:
- ✅ Loaded trained classifier + regressor models
- ✅ Applied identical feature engineering pipeline to inventory data
- ✅ Predicted Replacement Priority (4 classes) with confidence probabilities
- ✅ Predicted Asset Health Score (0-100 continuous)
- ✅ Computed ensemble Replacement Score (blended output)
- ✅ Applied color-coded priority labels (🔴🟠🟡🟢)
- ✅ Generated actionable recommendations per asset
- ✅ Exported `replacement_priority.csv` ranked by urgency
- ✅ Ready for Notebook 04 (Dashboard, Employee Priority, Procurement)